In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os
import pandas as pd
import numpy as np
from torchvision import datasets, transforms
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.metrics import f1_score
import random
import time
from torch.utils.data import TensorDataset
from torchvision.transforms import v2
from torch.utils.data.dataloader import default_collate

In [ ]:
cinic_mean_RGB = [0.47889522, 0.47227842, 0.43047404]
cinic_std_RGB  = [0.24205776, 0.23828046, 0.25874835]

SEEDS      = [42,123,2024] #[42, 123, 2024, 7, 999] (5 seeds were for baseline experiments, 3 seeds were for augmentation experiements and fewshot experiments)
NUM_EPOCHS = 7 #25 (25 was for baseline experiments and fewshot, 7 - for augmentation experiments)
DATA_PATH  = '/kaggle/input/datasets/mengcius/cinic10/'
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

# Ensuring reproducibility

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# Data loaders

In [ ]:
def preload_to_ram(dataset):
    loader = DataLoader(dataset, batch_size=512, num_workers=4, pin_memory=False)
    all_images, all_labels = [], []
    for images, labels in loader:
        all_images.append(images)
        all_labels.append(labels)
    return TensorDataset(torch.cat(all_images), torch.cat(all_labels))


from torch.utils.data import Subset, ConcatDataset, DataLoader

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
])

base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
])

image_aug_pipelines = {
    'flip': transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'blur': transforms.Compose([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'jitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutout': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.25, 0.25), ratio=(1, 1), value=0),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha_1': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha_4': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ])
}

batch_aug_pipelines = {
    'cutmix_alpha_1': transforms.v2.CutMix(num_classes=10, alpha=1.0),
    'cutmix_alpha_4': transforms.v2.CutMix(num_classes=10, alpha=4.0)
}

def train_collate_fn(aug_type, p=1.0):
    def collate_fn(batch):
        cutmix = batch_aug_pipelines[aug_type] 
        images, labels = default_collate(batch)
        if torch.rand(()) < p:
            images, labels = cutmix(images, labels)
        return images, labels
    return collate_fn

_loader_cache = {}

def limit_dataset(dataset, fraction):
    if fraction >= 1.0 or fraction <= 0.0:
        return dataset
    
    num_samples = int(len(dataset) * fraction)
    indices = torch.randperm(len(dataset))[:num_samples].tolist()
    return Subset(dataset, indices)

def get_few_shot_subset(dataset, samples_per_class: int):
    if samples_per_class is None:
        return dataset
        
    targets = np.array(dataset.targets)
    classes = np.unique(targets)
    
    few_shot_indices = []
    
    for cls in classes:
        cls_indices = np.where(targets == cls)[0]
        actual_samples = min(samples_per_class, len(cls_indices))
        selected_indices = np.random.choice(cls_indices, actual_samples, replace=False)
        few_shot_indices.extend(selected_indices)
        
    return Subset(dataset, few_shot_indices)

def reduce_dataset(dataset, data_fraction, samples_per_class):
    if samples_per_class is not None:
        return get_few_shot_subset(dataset, samples_per_class)
    elif data_fraction < 1.0:
        return limit_dataset(dataset, data_fraction)
    return dataset


def get_loaders(batch_size: int, path: str = DATA_PATH, aug_type: str = None, data_fraction: float = 1.0, samples_per_class: int = None):

    key = (batch_size, aug_type, data_fraction, samples_per_class)
    if key in _loader_cache:
        return _loader_cache[key]

    train_base = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=base_transform)
    valid_ds   = datasets.ImageFolder(root=os.path.join(path, 'valid'), transform=base_transform)
    test_ds    = datasets.ImageFolder(root=os.path.join(path, 'test'),  transform=base_transform)

    train_base = reduce_dataset(train_base, data_fraction, samples_per_class)
    valid_ds   = reduce_dataset(valid_ds, data_fraction, samples_per_class)
    test_ds    = reduce_dataset(test_ds, data_fraction, samples_per_class)

    valid_ds = preload_to_ram(valid_ds)
    test_ds  = preload_to_ram(test_ds)

    if aug_type is None:
        train_base = preload_to_ram(train_base)
        train_ds = train_base
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)
    
    else:
        if aug_type in ('cutmix_alpha_1', 'cutmix_alpha_4'): 
            train_clean_ds = preload_to_ram(train_base)
            train_cutmix_ds = preload_to_ram(train_base)

            train_clean_loader = DataLoader(
                train_clean_ds, batch_size=batch_size, shuffle=True,
                num_workers=4, pin_memory=True, persistent_workers=True
            )
            train_cutmix_loader = DataLoader(
                train_cutmix_ds, batch_size=batch_size, shuffle=True,
                num_workers=4, pin_memory=True, persistent_workers=True,
                collate_fn=train_collate_fn(aug_type)
            )
            train_loader = (train_clean_loader, train_cutmix_loader)
        else:
            if aug_type == 'random':
                chosen_transform = transforms.RandomChoice(list(image_aug_pipelines.values()))
            else:
                chosen_transform = image_aug_pipelines[aug_type]
                
            train_aug = datasets.ImageFolder(root=os.path.join(path, 'train'), transform=chosen_transform)
            train_aug = reduce_dataset(train_aug, data_fraction, samples_per_class)
                
            train_ds  = ConcatDataset([train_base, train_aug])
            train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=4, pin_memory=True, persistent_workers=True)

    valid_loader = DataLoader(valid_ds, batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True, persistent_workers=True)

    _loader_cache[key] = (train_loader, valid_loader, test_loader)
    return _loader_cache[key]

In this approach we adapt resnet-18 architecture for our dataset

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

def get_resnet18_for_cinic10(num_classes=10, use_pretrained=True, dropout=0.0, freeze_pretrained=True):

    weights = models.ResNet18_Weights.DEFAULT if use_pretrained else None
    model = models.resnet18(weights=weights)

    if use_pretrained and freeze_pretrained:
        for param in model.parameters():
            param.requires_grad = False 


    model.conv1 = nn.Conv2d(
        in_channels=3, 
        out_channels=64, 
        kernel_size=3, 
        stride=1, 
        padding=1, 
        bias=False
    )
    model.maxpool = nn.Identity()

    num_ftrs = model.fc.in_features
    if dropout > 0:
        model.fc = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(num_ftrs, num_classes)
        )
    else:
        model.fc = nn.Linear(num_ftrs, num_classes)

    return model

## Training

In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    all_preds, all_labels = [], []

    loaders_list = loader if isinstance(loader, tuple) else [loader]

    for current_loader in loaders_list:
        for images, labels in current_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            batch_size = images.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size
            
            preds = outputs.argmax(dim=1)
            all_preds.extend(preds.cpu().numpy())
            
            if labels.dim() > 1:
                labels_for_metric = labels.argmax(dim=1)
            else:
                labels_for_metric = labels
                
            all_labels.extend(labels_for_metric.cpu().numpy())

    avg_loss = total_loss / total_samples
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    
    return avg_loss, f1

In [ ]:
@torch.no_grad()
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    avg_loss = total_loss / len(loader.dataset)
    f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1

In [ ]:
def run_experiment(
    config_name: str,
    lr: float,
    batch_size: int,
    seeds: list = SEEDS,
    num_epochs: int = NUM_EPOCHS,
    dropout: float = 0.0,
    weight_decay: float = 0.0,
    aug_type: str = None,
    debug_size: int = None,
    data_fraction: float = 1.0,
    samples_per_class: int = None,
    freeze_pretrained: bool = True,
):
    """
    Runs a single hyperparameter configuration across all seeds.
    Returns a dict with mean ± std of validation and test F1.
    """
    print(f"\n{'='*60}")
    print(f"  Config: {config_name}")
    print(f"  LR={lr}  BS={batch_size}  dropout={dropout}  wd={weight_decay}")
    print(f"{'='*60}")

    val_f1_per_seed, test_f1_per_seed = [], []
    histories = []
    models = []

    for seed in seeds:
        set_seed(seed)
        print(f"\n  ── Seed {seed} ──")

        train_loader, valid_loader, test_loader = get_loaders(batch_size, aug_type=aug_type, data_fraction=data_fraction, samples_per_class=samples_per_class)

        model     = get_resnet18_for_cinic10(num_classes=10, use_pretrained=True, dropout=dropout, freeze_pretrained=freeze_pretrained).to(DEVICE)
        criterion = nn.CrossEntropyLoss()

        params_to_update = filter(lambda p: p.requires_grad, model.parameters())
        optimizer = optim.Adam(params_to_update, lr=lr, weight_decay=weight_decay)

        best_val_f1 = 0.0
        best_state  = None
        history     = []

        for epoch in range(1, num_epochs + 1):
            t0 = time.time()
            train_loss, train_f1 = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
            val_loss,   val_f1   = evaluate(model, valid_loader, criterion, DEVICE)
            elapsed = time.time() - t0

            history.append({
                'epoch': epoch, 'seed': seed, 'config': config_name,
                'train_loss': train_loss, 'train_f1': train_f1,
                'val_loss': val_loss,     'val_f1': val_f1,
            })

            if val_f1 >= best_val_f1:
                best_val_f1 = val_f1
                best_state  = {k: v.cpu().clone() for k, v in model.state_dict().items()}

            if epoch % 3 == 0 or epoch == num_epochs:
                print(f"    Epoch {epoch:2d}/{num_epochs} | "
                      f"Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f} | "
                      f"({elapsed:.1f}s)")

        model.load_state_dict(best_state)
        _, test_f1 = evaluate(model, test_loader, criterion, DEVICE)
        print(f"  → Best Val F1: {best_val_f1:.4f}  |  Test F1: {test_f1:.4f}")

        val_f1_per_seed.append(best_val_f1)
        test_f1_per_seed.append(test_f1)
        histories.extend(history)
        models.append(model)
    result = {
        'config':        config_name,
        'lr':            lr,
        'batch_size':    batch_size,
        'dropout':       dropout,
        'weight_decay':  weight_decay,
        'val_f1_mean':   np.mean(val_f1_per_seed),
        'val_f1_std':    np.std(val_f1_per_seed),
        'test_f1_mean':  np.mean(test_f1_per_seed),
        'test_f1_std':   np.std(test_f1_per_seed),
        'val_f1_seeds':  val_f1_per_seed,
        'test_f1_seeds': test_f1_per_seed,
    }

    print(f"\n Val  F1: {result['val_f1_mean']:.4f} ± {result['val_f1_std']:.4f}")
    print(f" Test F1: {result['test_f1_mean']:.4f} ± {result['test_f1_std']:.4f}")

    return result, pd.DataFrame(histories), models

# Learning rate testing

In [ ]:
BASELINE_BS = 64
LR_VALUES   = [1e-2, 1e-3, 1e-4]

phase1_results   = []
phase1_histories = []

for lr in LR_VALUES:
    cfg_name = f"LR={lr:.0e}_BS={BASELINE_BS}"
    res, hist, _ = run_experiment(cfg_name, lr=lr, batch_size=BASELINE_BS, debug_size=None, dropout=0.4)
    phase1_results.append(res)
    phase1_histories.append(hist)

phase1_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase1_results])

print("\n\n" + "─"*60)
print("PHASE 1 RESULTS — Learning Rate Sweep (BS=64)")
print("─"*60)
print(phase1_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

In [ ]:
phase1_df.to_csv('resnet18_lr_drop04_results.csv', index=False)
phase1_hist_df = pd.concat(phase1_histories, ignore_index=True)
phase1_hist_df.to_csv('resnet18_lr_drop04_history.csv', index=False)

# Batch size testing

In [ ]:
best_lr_idx = phase1_df['val_f1_mean'].idxmax()
best_lr     = phase1_df.loc[best_lr_idx, 'lr']

In [ ]:
best_lr = 1e-4

In [ ]:
BS_VALUES = [16, 32, 128]

phase2_results   = []
phase2_histories = []

for bs in BS_VALUES:
    cfg_name = f"LR={best_lr:.0e}_BS={bs}"
    res, hist, _ = run_experiment(cfg_name, lr=best_lr, batch_size=bs, debug_size=None)
    phase2_results.append(res)
    phase2_histories.append(hist)

phase2_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase2_results])

print("\n\n" + "─"*60)
print(f"PHASE 2 RESULTS — Batch Size Sweep (LR={best_lr:.0e})")
print("─"*60)
print(phase2_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))



In [ ]:
phase2_df.to_csv('resnet18_bs_results.csv', index=False)
phase2_hist_df = pd.concat(phase2_histories, ignore_index=True)
phase2_hist_df.to_csv('resnet18_bs_history.csv', index=False)

In [ ]:
best_batch_size = 16

# Dropout value testing

In [ ]:
DROPOUT_VALUES = [0.2, 0.3]
BASELINE_WD = 0.0

phase3_results   = []
phase3_histories = []

for dropoutval in DROPOUT_VALUES:
    cfg_name = f"LR={best_lr:.0e}_BS={best_batch_size}_DROPOUT={dropoutval}"
    res, hist, _ = run_experiment(cfg_name, lr=best_lr, batch_size=best_batch_size, dropout=dropoutval, debug_size=None)
    phase3_results.append(res)
    phase3_histories.append(hist)

phase3_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase3_results])

print("\n\n" + "─"*60)
print(f"PHASE 3 RESULTS — Dropout Size Sweep (LR={best_lr:.0e}, BS = {best_batch_size})")
print("─"*60)
print(phase3_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))



In [ ]:
phase3_df.to_csv('resnet18_dropout_results.csv', index=False)
phase3_hist_df = pd.concat(phase3_histories, ignore_index=True)
phase3_hist_df.to_csv('resnet18_dropout_history.csv', index=False)

In [ ]:
best_dropout = 0.2

# Weight decay value testing

In [ ]:
WEIGHT_DECAY_VALUES = [1e-5, 1e-4]

phase4_results   = []
phase4_histories = []

for weight_decay_val in WEIGHT_DECAY_VALUES:
    cfg_name = f"LR={best_lr:.0e}_BS={best_batch_size}_DROPOUT={best_dropout}_WEIGHT_DECAY={weight_decay_val:.0e}"
    res, hist, _ = run_experiment(cfg_name, lr=best_lr, batch_size=best_batch_size, dropout=best_dropout, weight_decay=weight_decay_val, debug_size=None)
    phase4_results.append(res)
    phase4_histories.append(hist)

phase4_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in phase4_results])

print("\n\n" + "─"*60)
print(f"PHASE 4 RESULTS — Weight-decay Size Sweep (LR={best_lr:.0e}, BS = {best_batch_size}, DROPOUT = {best_dropout})")
print("─"*60)
print(phase4_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

In [ ]:
phase4_df.to_csv('resnet18_weight_decay_results.csv', index=False)
phase4_hist_df = pd.concat(phase4_histories, ignore_index=True)
phase4_hist_df.to_csv('resnet18_weight_decay_history.csv', index=False)

In [ ]:
best_wd = 0.0

In [ ]:
def run_augmentation_benchmark(aug_type: str, data_fraction: float = 1.0, num_epochs: int = NUM_EPOCHS, freeze_pretrained: bool = True):

    results_list = []
    histories_list = []

    cfg_name = f"best_params_plus_augmentation_{aug_type}"
        
    res, hist, models = run_experiment(
        cfg_name, 
        lr=best_lr, 
        seeds = SEEDS,
        num_epochs = NUM_EPOCHS,
        batch_size=best_batch_size, 
        dropout=best_dropout, 
        weight_decay=best_wd, 
        aug_type=aug_type,
        data_fraction=data_fraction,
        freeze_pretrained=freeze_pretrained,
    )
    
    results_list.append(res)
    histories_list.append(hist)

    df_results = pd.DataFrame([{k: v for k, v in r.items() 
                                if k not in ('val_f1_seeds', 'test_f1_seeds')} 
                               for r in results_list])

    print("\n" + "─"*60)
    print(f" RESULTS")
    print("─"*60)
    display_cols = ['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']
    print(df_results[display_cols].to_string(index=False))
    
    return histories_list, df_results, models

# flip testing

In [ ]:
flip_histories, flip_df, flip_models = run_augmentation_benchmark(aug_type='flip', data_fraction=1, freeze_pretrained=False)

flip_df.to_csv('resnet18_flip_results.csv', index=False)
flip_hist_df = pd.concat(flip_histories, ignore_index=True)
flip_hist_df.to_csv('resnet18_flip_history.csv', index=False)

for i, model in enumerate(flip_models):
    name = f"model_flip_{i}.pth"
    torch.save(model.state_dict(), name)

# blur testing

In [ ]:
blur_histories, blur_df, blur_models = run_augmentation_benchmark(aug_type='blur', data_fraction=1, freeze_pretrained=False)

blur_df.to_csv('resnet18_blur_results.csv', index=False)
blur_hist_df = pd.concat(blur_histories, ignore_index=True)
blur_hist_df.to_csv('resnet18_blur_history.csv', index=False)

for i, model in enumerate(blur_models):
    name = f"model_blur_{i}.pth"
    torch.save(model.state_dict(), name)

# jitter testing

In [ ]:
jitter_histories, jitter_df, jitter_models = run_augmentation_benchmark(aug_type='jitter', data_fraction=1, freeze_pretrained=False)

jitter_df.to_csv('resnet18_jitter_results.csv', index=False)
jitter_hist_df = pd.concat(jitter_histories, ignore_index=True)
jitter_hist_df.to_csv('resnet18_jitter_history.csv', index=False)

for i, model in enumerate(jitter_models):
    name = f"model_jitter_{i}.pth"
    torch.save(model.state_dict(), name)

# cutout testing

In [ ]:
cutout_histories, cutout_df, cutout_models = run_augmentation_benchmark(aug_type='cutout', data_fraction=1, freeze_pretrained=False)

cutout_df.to_csv('resnet18_cutout_results.csv', index=False)
cutout_hist_df = pd.concat(cutout_histories, ignore_index=True)
cutout_hist_df.to_csv('resnet18_cutout_history.csv', index=False)

for i, model in enumerate(cutout_models):
    name = f"model_cutout_{i}.pth"
    torch.save(model.state_dict(), name)

# cutmix testing

In [ ]:
 cutmix_a_1_histories, cutmix_a_1_df, cutmix_a_1_models = run_augmentation_benchmark(aug_type='cutmix_alpha_1', data_fraction=1, freeze_pretrained=False)

cutmix_a_1_df.to_csv('resnet18_cutmix_alpha_1_results.csv', index=False)
cutmix_a_1_hist_df = pd.concat(cutmix_a_1_histories, ignore_index=True)
cutmix_a_1_hist_df.to_csv('resnet18_cutmix_alpha_1_history.csv', index=False)

for i, model in enumerate(cutmix_a_1_models):
    name = f"model_cutmix_alpha_1_{i}.pth"
    torch.save(model.state_dict(), name)

In [ ]:
cutmix_a_4_histories, cutmix_a_4_df, cutmix_a_4_models = run_augmentation_benchmark(aug_type='cutmix_alpha_4', data_fraction=1, freeze_pretrained=False)

cutmix_a_4_df.to_csv('resnet18_cutmix_alpha_4_results.csv', index=False)
cutmix_a_4_hist_df = pd.concat(cutmix_a_4_histories, ignore_index=True)
cutmix_a_4_hist_df.to_csv('resnet18_cutmix_alpha_4_history.csv', index=False)

for i, model in enumerate(cutmix_a_4_models):
    name = f"model_cutmix_alpha_4_{i}.pth"
    torch.save(model.state_dict(), name)

# random augmentations testing

In [ ]:
random_histories, random_df, random_models = run_augmentation_benchmark(aug_type='random', data_fraction=1, freeze_pretrained=False)

random_df.to_csv('resnet18_random_results.csv', index=False)
random_hist_df = pd.concat(random_histories, ignore_index=True)
random_hist_df.to_csv('resnet18_random_history.csv', index=False)

for i, model in enumerate(random_models):
    name = f"model_random_{i}.pth"
    torch.save(model.state_dict(), name)

# fewshot learning

In [ ]:
fewshot_results   = []
fewshot_histories = []
SAMPLES = [5,10,20]

for samples_per_class in SAMPLES:
    cfg_name = f"fewshot_{samples_per_class}_samples_per_class"
    res, hist, _ = run_experiment(cfg_name, lr=best_lr, batch_size=best_batch_size, dropout=best_dropout, weight_decay=best_wd, samples_per_class=samples_per_class, aug_type='flip', num_epochs=25)
    fewshot_results.append(res)
    fewshot_histories.append(hist)

fewshot_df = pd.DataFrame([{k: v for k, v in r.items()
                            if k not in ('val_f1_seeds', 'test_f1_seeds')}
                           for r in fewshot_results])

print("\n\n" + "─"*60)
print(f"FEWSHOT RESULTS")
print("─"*60)
print(fewshot_df[['config', 'val_f1_mean', 'val_f1_std', 'test_f1_mean', 'test_f1_std']].to_string(index=False))

In [ ]:
fewshot_df.to_csv('resnet18_fewshot_results.csv', index=False)
fewshot_hist_df = pd.concat(fewshot_histories, ignore_index=True)
fewshot_hist_df.to_csv('resnet18_fewshot_history.csv', index=False)